# Azure Retail Pipeline — PySpark Transformation Notebook

**Author:** Daya Meenakshi Bala Subbu  
**Project:** Azure Retail Customer Purchase Data Pipeline  
**GitHub:** https://github.com/DayaMeenakshiBalaSubbu/azure-retail-data-pipeline

---

## Setup Instructions

Before running this notebook replace the placeholder values below:

| Placeholder | Replace With |
|---|---|
| `<YOUR_STORAGE_ACCOUNT_NAME>` | Your ADLS Gen2 storage account name |
| `<YOUR_STORAGE_ACCOUNT_KEY>` | Your storage account key from Azure Portal |

> Never commit real credentials to GitHub!
> Get your key from: Azure Portal → Storage Account → Access keys

---

## Pipeline Overview

This notebook implements the Bronze to Gold transformation layer:

```
Bronze (raw CSV files)
        ↓
PySpark Transformations (this notebook)
        ↓
Gold Zone (clean Parquet files)
        ↓
Databricks SQL Tables (retail_gold)
```

## Transformations Applied

1. Schema casting — integer, double, date types
2. Derived column — total_amount = quantity x unit_price
3. String normalisation — uppercase SKUs, lowercase emails
4. Deduplication — dropDuplicates on primary keys
5. Null removal — dropna on critical fields
6. Dataset union — combine store + online transactions


In [ ]:
#Configure storage access
#This connects Databricks to ADLS Gen2 storage

storage_account_name = "<YOUR_STORAGE_ACCOUNT_NAME>"
storage_account_key  = "<YOUR_STORAGE_ACCOUNT_KEY>"

#Set Spark configuration to access ADLS Gen2
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("Storage connected successfully!")
print(f"Storage account: {storage_account_name}")


In [ ]:
#Read all 3 source files from Bronze
storage = "<YOUR_STORAGE_ACCOUNT_NAME>"
bronze = f"abfss://bronze@{storage}.dfs.core.windows.net"

#Read store transactions
df_store = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{bronze}/store/transactions.csv")

#Read online purchases
df_online = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{bronze}/online/purchases.csv")

#Read customer demographics
df_customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{bronze}/customers/demographics.csv")

print("All 3 files loaded from Bronze!")
print(f"Store transactions : {df_store.count()} rows")
print(f"Online purchases   : {df_online.count()} rows")
print(f"Customers          : {df_customers.count()} rows")

In [ ]:
#Clean store transactions
from pyspark.sql import functions as F

df_store_clean = df_store \
    .withColumn("quantity",
        F.col("quantity").cast("integer")) \
    .withColumn("unit_price",
        F.col("unit_price").cast("double")) \
    .withColumn("transaction_date",
        F.to_date("transaction_date", "yyyy-MM-dd")) \
    .withColumn("total_amount",
        F.round(F.col("quantity") * F.col("unit_price"), 2)) \
    .withColumn("product_sku",
        F.upper(F.trim(F.col("product_sku")))) \
    .withColumn("source_system", F.lit("store_pos")) \
    .dropDuplicates(["transaction_id"]) \
    .dropna(subset=["transaction_id", "customer_id"])

print("✅ Store transactions cleaned!")
print(f"Rows after cleaning: {df_store_clean.count()}")
df_store_clean.show(5)

In [ ]:
# Clean online purchases
df_online_clean = df_online \
    .withColumn("quantity",
        F.col("quantity").cast("integer")) \
    .withColumn("unit_price",
        F.col("unit_price").cast("double")) \
    .withColumn("order_date",
        F.to_date("order_date", "yyyy-MM-dd")) \
    .withColumn("total_amount",
        F.round(F.col("quantity") * F.col("unit_price"), 2)) \
    .withColumn("product_sku",
        F.upper(F.trim(F.col("product_sku")))) \
    .withColumn("source_system", F.lit("online")) \
    .withColumnRenamed("order_id", "transaction_id") \
    .withColumnRenamed("order_date", "transaction_date") \
    .dropDuplicates(["transaction_id"]) \
    .dropna(subset=["transaction_id", "customer_id"])

print("Online purchases cleaned!")
print(f"Rows after cleaning: {df_online_clean.count()}")
df_online_clean.show(5)

In [ ]:
# Clean customer demographics
df_customers_clean = df_customers \
    .withColumn("join_date",
        F.to_date("join_date", "yyyy-MM-dd")) \
    .withColumn("loyalty_tier",
        F.upper(F.trim(F.col("loyalty_tier")))) \
    .withColumn("name",
        F.initcap(F.trim(F.col("name")))) \
    .withColumn("email",
        F.lower(F.trim(F.col("email")))) \
    .withColumn("email_valid",
        F.col("email").rlike("^[^@]+@[^@]+\\.[^@]+$")) \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id"])

print("Customer demographics cleaned!")
print(f"Rows after cleaning: {df_customers_clean.count()}")
df_customers_clean.show(5)

In [ ]:
# Combining store and online transactions
common_cols = [
    "transaction_id", "customer_id", "product_sku",
    "quantity", "unit_price", "total_amount",
    "transaction_date", "source_system"
]

df_all_transactions = df_store_clean.select(common_cols) \
    .unionByName(df_online_clean.select(common_cols))

print("All transactions combined!")
print(f"Total rows: {df_all_transactions.count()}")
print(f"Store rows: {df_store_clean.count()}")
print(f"Online rows: {df_online_clean.count()}")
df_all_transactions.show(5)

In [ ]:
# Write clean data to Gold zone
storage = "<YOUR_STORAGE_ACCOUNT_NAME>"
gold = f"abfss://gold@{storage}.dfs.core.windows.net"

# Write all transactions to Gold
df_all_transactions.write \
    .mode("overwrite") \
    .option("header", True) \
    .partitionBy("source_system") \
    .parquet(f"{gold}/transactions/")

# Write customers to Gold
df_customers_clean.write \
    .mode("overwrite") \
    .option("header", True) \
    .parquet(f"{gold}/customers/")

print("Gold zone loaded successfully!")
print(f"Transactions written: {df_all_transactions.count()} rows")
print(f"Customers written   : {df_customers_clean.count()} rows")

In [ ]:
# Verify Gold zone contents
print("📁 Gold zone contents:")
for f in dbutils.fs.ls(f"abfss://gold@{storage}.dfs.core.windows.net/"):
    print(f"  {f.path}")

print("\n Gold/transactions/ contents:")
for f in dbutils.fs.ls(f"abfss://gold@{storage}.dfs.core.windows.net/transactions/"):
    print(f"  {f.path}")

In [ ]:
# Quick analytics on Gold data
# Read back from Gold to verify
df_gold = spark.read.parquet(
    f"abfss://gold@{storage}.dfs.core.windows.net/transactions/"
)

print("Gold data verified!")
print(f"Total rows: {df_gold.count()}")
print("\n Revenue by source channel:")
df_gold.groupBy("source_system") \
    .agg(
        F.count("transaction_id").alias("transactions"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ) \
    .show()

print("\n Top 5 products by revenue:")
df_gold.groupBy("product_sku") \
    .agg(F.round(F.sum("total_amount"), 2).alias("revenue")) \
    .orderBy(F.desc("revenue")) \
    .show(5)

In [ ]:
# Reload everything from scratch
from pyspark.sql import functions as F

storage = "<YOUR_STORAGE_ACCOUNT_NAME>"
bronze = f"abfss://bronze@{storage}.dfs.core.windows.net"

# Step 1 — Set storage access
spark.conf.set(
    f"fs.azure.account.key.{storage}.dfs.core.windows.net",
    "<YOUR_STORAGE_ACCOUNT_KEY>"
)

# Step 2 — Read all 3 files
df_store = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{bronze}/store/transactions.csv")

df_online = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{bronze}/online/purchases.csv")

df_customers = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{bronze}/customers/demographics.csv")

# Step 3 — Clean store transactions
df_store_clean = df_store \
    .withColumn("quantity", F.col("quantity").cast("integer")) \
    .withColumn("unit_price", F.col("unit_price").cast("double")) \
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd")) \
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2)) \
    .withColumn("product_sku", F.upper(F.trim(F.col("product_sku")))) \
    .withColumn("source_system", F.lit("store_pos")) \
    .dropDuplicates(["transaction_id"]) \
    .dropna(subset=["transaction_id", "customer_id"])

# Step 4 — Clean online purchases
df_online_clean = df_online \
    .withColumn("quantity", F.col("quantity").cast("integer")) \
    .withColumn("unit_price", F.col("unit_price").cast("double")) \
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd")) \
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2)) \
    .withColumn("product_sku", F.upper(F.trim(F.col("product_sku")))) \
    .withColumn("source_system", F.lit("online")) \
    .withColumnRenamed("order_id", "transaction_id") \
    .withColumnRenamed("order_date", "transaction_date") \
    .dropDuplicates(["transaction_id"]) \
    .dropna(subset=["transaction_id", "customer_id"])

# Step 5 — Clean customers
df_customers_clean = df_customers \
    .withColumn("join_date", F.to_date("join_date", "yyyy-MM-dd")) \
    .withColumn("loyalty_tier", F.upper(F.trim(F.col("loyalty_tier")))) \
    .withColumn("name", F.initcap(F.trim(F.col("name")))) \
    .withColumn("email", F.lower(F.trim(F.col("email")))) \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id"])

# Step 6 — Combine transactions
common_cols = ["transaction_id", "customer_id", "product_sku",
               "quantity", "unit_price", "total_amount",
               "transaction_date", "source_system"]

df_all_transactions = df_store_clean.select(common_cols) \
    .unionByName(df_online_clean.select(common_cols))

print("✅ All data reloaded!")
print(f"Store transactions : {df_store_clean.count()} rows")
print(f"Online purchases   : {df_online_clean.count()} rows")
print(f"Customers          : {df_customers_clean.count()} rows")
print(f"Total transactions : {df_all_transactions.count()} rows")

In [ ]:
# Create retail database in Databricks
spark.sql("CREATE DATABASE IF NOT EXISTS retail_gold")
spark.sql("USE retail_gold")
print("Database created!")

In [ ]:
# Create database and save tables
spark.sql("CREATE DATABASE IF NOT EXISTS retail_gold")

# Save transactions
df_all_transactions.write \
    .mode("overwrite") \
    .saveAsTable("retail_gold.fact_transactions")

print(f" fact_transactions saved! Rows: {df_all_transactions.count()}")

# Save customers
df_customers_clean.write \
    .mode("overwrite") \
    .saveAsTable("retail_gold.dim_customer")

print(f" dim_customer saved! Rows: {df_customers_clean.count()}")

In [ ]:
# Run business analytics queries
print("=" * 50)
print("RETAIL ANALYTICS DASHBOARD")
print("=" * 50)

# Query 1 — Revenue by channel
print("\n1 Revenue by Channel:")
spark.sql("""
    SELECT 
        source_system AS channel,
        COUNT(*) AS total_transactions,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM retail_gold.fact_transactions
    GROUP BY source_system
    ORDER BY total_revenue DESC
""").show()

# Query 2 — Top 10 products
print("2 Top 10 Products by Revenue:")
spark.sql("""
    SELECT 
        product_sku,
        COUNT(*) AS times_sold,
        SUM(quantity) AS total_units,
        ROUND(SUM(total_amount), 2) AS revenue
    FROM retail_gold.fact_transactions
    GROUP BY product_sku
    ORDER BY revenue DESC
    LIMIT 10
""").show()

# Query 3 — Customer loyalty analysis
print("3 Revenue by Customer Loyalty Tier:")
spark.sql("""
    SELECT 
        c.loyalty_tier,
        COUNT(DISTINCT t.customer_id) AS customers,
        COUNT(t.transaction_id) AS transactions,
        ROUND(SUM(t.total_amount), 2) AS total_revenue
    FROM retail_gold.fact_transactions t
    JOIN retail_gold.dim_customer c
    ON t.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
    ORDER BY total_revenue DESC
""").show()

# Query 4 — Monthly revenue trend
print("4 Monthly Revenue Trend:")
spark.sql("""
    SELECT 
        YEAR(transaction_date) AS year,
        MONTH(transaction_date) AS month,
        COUNT(*) AS transactions,
        ROUND(SUM(total_amount), 2) AS monthly_revenue
    FROM retail_gold.fact_transactions
    GROUP BY YEAR(transaction_date), MONTH(transaction_date)
    ORDER BY year, month
""").show()


In [ ]:
# Export retail data to download
import pandas as pd

df_full = spark.sql("""
    SELECT 
        t.transaction_id,
        t.customer_id,
        t.product_sku,
        t.quantity,
        t.unit_price,
        t.total_amount,
        t.transaction_date,
        t.source_system AS channel,
        c.loyalty_tier,
        c.city,
        c.state
    FROM retail_gold.fact_transactions t
    LEFT JOIN retail_gold.dim_customer c
    ON t.customer_id = c.customer_id
""").toPandas()

# Display the data
print(f"Total rows: {len(df_full)}")
display(df_full)
